# 🐾 喵撲輸入法純淨語料擴增與上下文推理訓練一鍵管線（總預算嚴格 ≤ $15 USD）
- **100% 純淨合規授權**：僅使用自產 `MIT` 領域詞庫與語料（`gemini-3.1-flash-lite` 合成）、以及既有的 `mimo-pairs` 核准來源（Tatoeba `CC-BY-2.0-FR`、Common Voice `CC0`、CrossWOZ/KdConv `Apache-2.0`、MASSIVE `CC-BY-4.0`）。**零網頁爬蟲、零新聞語料、零 CC BY-SA**。
- **嚴格費用硬斷路器**：`vertex_live_synth.py` 跨階段累積追蹤實付帳本，硬上限定在 `$14.50 USD`（< `$15.00`）。
- **零評估洩漏**：所有生成與計數管線均自動排除 `Tests/eval/*.tsv` 任意繁簡 `>= 5` 字子句。

In [ ]:
from google.colab import drive, auth
drive.mount('/content/drive')
auth.authenticate_user()

%pip install -q --upgrade torch certifi onnx onnxruntime
REPO_DIR = '/content/Meowpuut-Core'
GCP_PROJECT = 'gen-lang-client-0285872710'
WORK_DIR = '/content/drive/MyDrive/meowpuut-colab-work'
!mkdir -p {WORK_DIR}/shards {WORK_DIR}/tables {WORK_DIR}/reranker

## 步驟 1（約 $0.15 USD）：自產現代與專業詞彙擴充

In [ ]:
%cd {REPO_DIR}
!python3 scripts/colab_lexicon_mine.py export-modern-seed
!python3 scripts/vertex_live_synth.py gen-terms --project {GCP_PROJECT} --workers 20 --max-budget-usd 1.00 --out DataSources/mimo-domains/modern-llm-terms.tsv
!python3 scripts/colab_lexicon_mine.py merge-terms --apply

## 步驟 2（≤ $13.50 USD）：孤兒詞錨點 ＋ 高密度現代與消歧語料合成

In [ ]:
!python3 scripts/colab_lexicon_mine.py export-orphans --lexicon-dir .build/lexicon-stage1 --include-lexicon-orphans --group-size 8 --max-unique-anchors 24000
!python3 scripts/colab_lexicon_mine.py plan-corpus --max-anchor-tasks 5800
!python3 scripts/vertex_live_synth.py gen-corpus --tasks .build/colab-corpus/vertex_corpus_tasks.jsonl --out .build/colab-corpus/vertex_corpus_raw.jsonl --workers 24 --max-budget-usd 13.50
!python3 scripts/vertex_live_synth.py ingest-corpus --raw .build/colab-corpus/vertex_corpus_raw.jsonl --out-dir DataSources/mimo-everyday --lexicon-dir .build/lexicon-stage1

## 步驟 3：重算均衡 `mimo_pairs` 與重建 `mimo_base.bin` ＋ Before/After 評測

In [ ]:
!python3 scripts/mimo_pairs.py fetch
!python3 scripts/mimo_pairs.py count .build/lexicon-stage1 --exclude Tests/eval/heldout-sentences.tsv --exclude Tests/eval/test2-sentences.tsv --exclude Tests/eval/modern-stress-sentences.tsv --jobs 6
!python3 scripts/mimo_pairs.py snapshot .build/lexicon-stage1 --keep kl:2000000
!bash scripts/build-lexicon.sh .build/lexicon-v2
!python3 scripts/eval-ranking.py Tests/eval/heldout-sentences.tsv --resources .build/lexicon-v2
!python3 scripts/eval-ranking.py Tests/eval/modern-stress-sentences.tsv --resources .build/lexicon-v2 --show-failures 15

## 步驟 4（$0 GPU）：建置橋接 Trigram/Skip-1 與訓練 3–4 MB INT8 上下文重排器

In [ ]:
!python3 scripts/colab_corpus_mix.py export-clean-lines --lexicon-dir .build/lexicon-v2 --out-dir {WORK_DIR}/shards
!python3 scripts/colab_corpus_mix.py build-context-tables --lexicon-dir .build/lexicon-v2 --corpus-dirs DataSources/mimo-everyday {WORK_DIR}/shards --out-dir {WORK_DIR}/tables
!python3 scripts/colab_train_reranker.py train --corpus-dirs DataSources/mimo-everyday {WORK_DIR}/shards --out-dir {WORK_DIR}/reranker --epochs 3 --batch-size 256 --max-lines 1200000
!python3 scripts/eval-context-rerank.py --resources .build/lexicon-v2 --tables-dir {WORK_DIR}/tables --model-dir {WORK_DIR}/reranker